### 1.0 - Libraries 

In [1]:
import numpy as np
from scipy.optimize import minimize
from qiskit.circuit.library import qaoa_ansatz
from qiskit.primitives import StatevectorSampler
from qiskit.quantum_info import SparsePauliOp

### 2.0 - Data

In [2]:
# 1. Define your QUBO matrix Q (Goal: minimize x^T @ Q @ x, where x in {0, 1}^n)
# Example: 3 variables
# Objective: -2*x0 - 2*x1 - 3*x2 + 4*x0*x1 + 2*x1*x2
# Optimal solution is x = [1, 0, 1] with cost = -5
Q = np.array([
    [-2.0,  2.0,  0.0],
    [ 2.0, -2.0,  1.0],
    [ 0.0,  1.0, -3.0]
])

Q = 0.5 * (Q + Q.T)

In [3]:
# Symmetrize Q just in case upper-triangular input is provided
num_qubits = Q.shape[0]

### 3.0 - QUBO to Hamiltonian

In [4]:
# Convert QUBO matrix Q to Ising Hamiltonian (SparsePauliOp)
# Substitution: x_i = (1 - Z_i) / 2
def qubo_to_ising(Q_mat):
    n = Q_mat.shape[0]
    pauli_list = []
    offset = 0.0

    # Linear Z terms and constant offset
    for i in range(n):
        h_i = -0.5 * np.sum(Q_mat[i, :])
        offset += 0.5 * Q_mat[i, i]
        if abs(h_i) > 1e-10:
            # Qiskit uses little-endian ordering (qubit 0 is rightmost)
            label = ["I"] * n
            label[n - 1 - i] = "Z"
            pauli_list.append(("".join(label), h_i))

    # Quadratic ZZ interaction terms
    for i in range(n):
        for j in range(i + 1, n):
            J_ij = 0.5 * Q_mat[i, j]
            offset += 0.5 * Q_mat[i, j]
            if abs(J_ij) > 1e-10:
                label = ["I"] * n
                label[n - 1 - i] = "Z"
                label[n - 1 - j] = "Z"
                pauli_list.append(("".join(label), J_ij))

    return SparsePauliOp.from_list(pauli_list), offset

cost_hamiltonian, offset = qubo_to_ising(Q)


### 3.5 - Hamiltonian to Circuit

In [5]:
# Build the QAOA circuit (reps = p layers)
reps = 2
ansatz = qaoa_ansatz(cost_operator=cost_hamiltonian, reps=reps)
ansatz.measure_all()

### 4.0 - Functions 

In [6]:
# Classical QUBO evaluation helper
def evaluate_qubo(bitstring):
    # Reverse bitstring to match [x0, x1, ..., xn-1]
    x = np.array([int(b) for b in bitstring[::-1]])
    return float(x @ Q @ x)


In [7]:
# Cost function using V2 StatevectorSampler
sampler = StatevectorSampler(seed=42)

def cost_func(params):
    # V2 Sampler accepts PUBs: (circuit, parameter_values)
    job = sampler.run([(ansatz, params)], shots=2048)
    counts = job.result()[0].data.meas.get_counts()
    total_shots = sum(counts.values())
    
    expectation = sum(evaluate_qubo(bits) * count for bits, count in counts.items()) / total_shots
    return expectation


In [8]:
# 6. Optimize QAOA parameters (gamma and beta)
init_params = np.random.uniform(0, np.pi, ansatz.num_parameters)
result = minimize(cost_func, init_params, method="COBYLA", options={"maxiter": 200})

### 5.0 - Run Circuit 

In [9]:
# Sample final optimized circuit
final_job = sampler.run([(ansatz, result.x)], shots=2048)
final_counts = final_job.result()[0].data.meas.get_counts()
sorted_counts = sorted(final_counts.items(), key=lambda item: item[1], reverse=True)


### 8.0 - Post Processing 

In [10]:
print(f"Minimum expectation value found: {result.fun:.4f}\n")
print("Top sampled solutions [x0, x1, x2]:")
for bitstring, count in sorted_counts[:5]:
    x_vec = [int(b) for b in bitstring[::-1]]
    cost = evaluate_qubo(bitstring)
    print(f"  x = {x_vec} | Shots: {count:4d} | QUBO Cost: {cost:.2f}")

Minimum expectation value found: -3.5522

Top sampled solutions [x0, x1, x2]:
  x = [1, 0, 1] | Shots:  953 | QUBO Cost: -5.00
  x = [0, 1, 1] | Shots:  692 | QUBO Cost: -3.00
  x = [1, 1, 0] | Shots:  164 | QUBO Cost: 0.00
  x = [1, 0, 0] | Shots:  118 | QUBO Cost: -2.00
  x = [0, 1, 0] | Shots:   86 | QUBO Cost: -2.00
